In [1]:
#Loan Prediction with Classification

In [2]:
import pandas as pd
pd.set_option("display.max_columns",55)

import warnings
warnings.filterwarnings("ignore")
import seaborn as sns
import miceforest as mf 
from sklearn.linear_model import LinearRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB, BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

def classification_test(x, y):
    # Bütün modelleri tanımlıyorum
    models = {
        "BernoulliNB": BernoulliNB(),
        "GaussianNB": GaussianNB(),
        "Logistic Regression": LogisticRegression(),
        "Decision Tree": DecisionTreeClassifier(),
        "Random Forest": RandomForestClassifier(),
        "Gradient Boosting": GradientBoostingClassifier(),
        "K-Neighbors": KNeighborsClassifier()
    }
    
    # Eğitim ve test verisini ayır
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
    
    # Sonuçları saklamak için boş dataframe
    results = pd.DataFrame(columns=['Accuracy'], index=models.keys())
    classification_reports = {}
    
    for name, model in models.items():
        model.fit(x_train, y_train)
        y_pred = model.predict(x_test)
        
        # Accuracy hesapla
        acc = accuracy_score(y_test, y_pred)
        results.loc[name, 'Accuracy'] = acc
        
        # Classification report sakla
        classification_reports[name] = classification_report(y_test, y_pred, output_dict=True)
    
    return results.sort_values(by='Accuracy', ascending=False), classification_reports

In [4]:
df=pd.read_csv("LoansTrainingSet.csv")

In [5]:
#EDA

In [6]:
df.head()

,Loan ID,Customer ID,Loan Status,Current Loan Amount,Term,Credit Score,Years in current job,Home Ownership,Annual Income,Purpose,Monthly Debt,Years of Credit History,Months since last delinquent,Number of Open Accounts,Number of Credit Problems,Current Credit Balance,Maximum Open Credit,Bankruptcies,Tax Liens
0,000025bb-5694-4cff-b17d-192b1a98ba44,5ebc8bb1-5eb9-4404-b11b-a6eebc401a19,Fully Paid,11520,Short Term,741.0,10+ years,Home Mortgage,33694.0,Debt Consolidation,$584.03,12.3,41.0,10,0,6760,16056,0.0,0.0
1,00002c49-3a29-4bd4-8f67-c8f8fbc1048c,927b388d-2e01-423f-a8dc-f7e42d668f46,Fully Paid,3441,Short Term,734.0,4 years,Home Mortgage,42269.0,other,"$1,106.04",26.3,NaN,17,0,6262,19149,0.0,0.0
2,00002d89-27f3-409b-aa76-90834f359a65,defce609-c631-447d-aad6-1270615e89c4,Fully Paid,21029,Short Term,747.0,10+ years,Home Mortgage,90126.0,Debt Consolidation,"$1,321.85",28.8,NaN,5,0,20967,28335,0.0,0.0
3,00005222-b4d8-45a4-ad8c-186057e24233,070bcecb-aae7-4485-a26a-e0403e7bb6c5,Fully Paid,18743,Short Term,747.0,10+ years,Own Home,38072.0,Debt Consolidation,$751.92,26.2,NaN,9,0,22529,43915,0.0,0.0
4,0000757f-a121-41ed-b17b-162e76647c1f,dde79588-12f0-4811-bab0-e2b07f633fcd,Fully Paid,11731,Short Term,746.0,4 years,Rent,50025.0,Debt Consolidation,$355.18,11.5,NaN,12,0,17391,37081,0.0,0.0


In [7]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Current Loan Amount,256984.0,1.371331e+07,3.438131e+07,505.0,8299.0,14298.0,24367.0,99999999.0
Credit Score,195308.0,1.251116e+03,1.762017e+03,585.0,714.0,733.0,744.0,7510.0
Annual Income,195308.0,7.195272e+04,5.887757e+04,0.0,44321.0,61242.0,86462.0,8713547.0
Years of Credit History,256984.0,1.829019e+01,7.075747e+00,3.4,13.5,17.0,21.7,70.5
Months since last delinquent,116601.0,3.488145e+01,2.185417e+01,0.0,16.0,32.0,51.0,176.0
Number of Open Accounts,256984.0,1.110627e+01,4.982982e+00,0.0,8.0,10.0,14.0,76.0
Number of Credit Problems,256984.0,1.566284e-01,4.607309e-01,0.0,0.0,0.0,0.0,11.0
Current Credit Balance,256984.0,1.540656e+04,1.966506e+04,0.0,5974.0,11078.0,19319.0,1731412.0
Bankruptcies,256455.0,1.103156e-01,3.362287e-01,0.0,0.0,0.0,0.0,7.0
Tax Liens,256961.0,2.720257e-02,2.459499e-01,0.0,0.0,0.0,0.0,11.0


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 256984 entries, 0 to 256983
Data columns (total 19 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   Loan ID                       256984 non-null  object 
 1   Customer ID                   256984 non-null  object 
 2   Loan Status                   256984 non-null  object 
 3   Current Loan Amount           256984 non-null  int64  
 4   Term                          256984 non-null  object 
 5   Credit Score                  195308 non-null  float64
 6   Years in current job          245508 non-null  object 
 7   Home Ownership                256984 non-null  object 
 8   Annual Income                 195308 non-null  float64
 9   Purpose                       256984 non-null  object 
 10  Monthly Debt                  256984 non-null  object 
 11  Years of Credit History       256984 non-null  float64
 12  Months since last delinquent  116601 non-nul

In [9]:
df.isnull().sum()

Loan ID                              0
Customer ID                          0
Loan Status                          0
Current Loan Amount                  0
Term                                 0
Credit Score                     61676
Years in current job             11476
Home Ownership                       0
Annual Income                    61676
Purpose                              0
Monthly Debt                         0
Years of Credit History              0
Months since last delinquent    140383
Number of Open Accounts              0
Number of Credit Problems            0
Current Credit Balance               0
Maximum Open Credit                  0
Bankruptcies                       529
Tax Liens                           23
dtype: int64

In [10]:
df["Loan ID"].value_counts()

Loan ID
3f6bd37a-b0bc-4d85-93c7-eea53df601fb    4
65b03871-1531-46ed-b805-a5df41477f03    4
744f26f5-685b-4c1d-89cc-8f32fef69373    4
7902a3ce-5054-4192-bf2f-bb597f3f870c    4
71121052-b403-4ec8-b3fc-25bb3223ed0d    4
                                       ..
5e122f17-d56f-4785-9912-a1a717b180e1    1
5e127563-cd0b-408c-9fc2-8066c4a3e965    1
5e12efa6-0bec-4b0d-bcea-454fa1ebc9f4    1
5e12f797-9600-4f00-965f-b76be5108331    1
ffffe32e-ed17-459f-9cfd-7b9ee7972933    1
Name: count, Length: 215700, dtype: int64

In [11]:
df["Loan ID"].value_counts().value_counts()

count
1    176520
2     38118
4      1042
3        20
Name: count, dtype: int64

In [12]:
df[df['Loan ID'] == '65b03871-1531-46ed-b805-a5df41477f03']

,Loan ID,Customer ID,Loan Status,Current Loan Amount,Term,Credit Score,Years in current job,Home Ownership,Annual Income,Purpose,Monthly Debt,Years of Credit History,Months since last delinquent,Number of Open Accounts,Number of Credit Problems,Current Credit Balance,Maximum Open Credit,Bankruptcies,Tax Liens
102269,65b03871-1531-46ed-b805-a5df41477f03,abbb7bf6-908f-4500-bc46-a6b1b87d3d8a,Charged Off,7576,Short Term,7160.0,8 years,Home Mortgage,70476.0,Debt Consolidation,"$1,544.60",28.6,NaN,21,0,28302,86551,0.0,0.0
102270,65b03871-1531-46ed-b805-a5df41477f03,abbb7bf6-908f-4500-bc46-a6b1b87d3d8a,Charged Off,7677,Short Term,NaN,8 years,Home Mortgage,NaN,Debt Consolidation,"$1,565.16",28.6,NaN,21,0,28679,87703,0.0,0.0
102271,65b03871-1531-46ed-b805-a5df41477f03,abbb7bf6-908f-4500-bc46-a6b1b87d3d8a,Charged Off,7576,Short Term,716.0,8 years,Home Mortgage,70476.0,Debt Consolidation,"$1,544.60",28.6,NaN,21,0,28302,86551,0.0,0.0
102272,65b03871-1531-46ed-b805-a5df41477f03,abbb7bf6-908f-4500-bc46-a6b1b87d3d8a,Charged Off,7677,Short Term,716.0,8 years,Home Mortgage,71414.0,Debt Consolidation,"$1,565.16",28.6,NaN,21,0,28679,87703,0.0,0.0


In [13]:
df_unique = df.drop_duplicates(subset=['Loan ID'])

In [14]:
# There was same loans multiple times, we cleared

In [15]:
df_unique["Loan ID"].value_counts()

Loan ID
000025bb-5694-4cff-b17d-192b1a98ba44    1
aa9461f5-1536-41f9-be2b-ff8a210932d8    1
aa8e9502-8b68-4480-8d5d-6203f617c9b1    1
aa8f5ce4-76f3-4fae-9a07-58cc65d90271    1
aa8fa68c-960a-40c4-9a7c-04db7bb92e1b    1
                                       ..
55527f06-c930-4fe1-bbcb-3610ae3649ed    1
5552da49-009b-4a24-9f37-b0fe36cd48aa    1
555312ca-ddc5-4814-9d88-e01c1986f708    1
555408d1-c1cf-47a0-8b1e-1e42a0e40f1a    1
ffffe32e-ed17-459f-9cfd-7b9ee7972933    1
Name: count, Length: 215700, dtype: int64

In [16]:
df_unique["Loan Status"].value_counts()

Loan Status
Fully Paid     176191
Charged Off     39509
Name: count, dtype: int64

In [17]:
df_unique["Term"].value_counts()

Term
Short Term    166523
Long Term      49177
Name: count, dtype: int64

In [18]:
df_unique["Years in current job"].value_counts()

Years in current job
10+ years    66711
2 years      19831
< 1 year     17544
3 years      17428
5 years      14987
1 year       14130
4 years      13632
6 years      12230
7 years      11713
8 years      10232
9 years       8272
Name: count, dtype: int64

In [19]:
df_unique["Years in current job"] = df_unique["Years in current job"].replace("10+ years", "15")
df_unique["Years in current job"] = df_unique["Years in current job"].replace("< 1 year", "0")

In [20]:
df_unique["Years in current job"].value_counts()

Years in current job
15         66711
2 years    19831
0          17544
3 years    17428
5 years    14987
1 year     14130
4 years    13632
6 years    12230
7 years    11713
8 years    10232
9 years     8272
Name: count, dtype: int64

In [21]:
df_unique["Home Ownership"].value_counts()

Home Ownership
Home Mortgage    106492
Rent              89619
Own Home          19094
HaveMortgage        495
Name: count, dtype: int64

In [22]:
df_unique["Home Ownership"] = df_unique["Home Ownership"].replace("HaveMortgage", "Home Mortgage")

In [23]:
df_unique["Home Ownership"].value_counts()

Home Ownership
Home Mortgage    106987
Rent              89619
Own Home          19094
Name: count, dtype: int64

In [24]:
df_unique["Purpose"].value_counts()

Purpose
Debt Consolidation      171096
Home Improvements        12840
other                    11726
Other                     8281
Business Loan             3611
Buy a Car                 2926
Medical Bills             2375
Take a Trip               1312
Buy House                 1308
Educational Expenses       225
Name: count, dtype: int64

In [25]:
df_unique["Purpose"] = df_unique["Purpose"].replace("other", "Other")

In [26]:
df_unique.isnull().sum()

Loan ID                              0
Customer ID                          0
Loan Status                          0
Current Loan Amount                  0
Term                                 0
Credit Score                     55008
Years in current job              8990
Home Ownership                       0
Annual Income                    55008
Purpose                              0
Monthly Debt                         0
Years of Credit History              0
Months since last delinquent    118262
Number of Open Accounts              0
Number of Credit Problems            0
Current Credit Balance               0
Maximum Open Credit                  0
Bankruptcies                       452
Tax Liens                           22
dtype: int64

In [27]:
df_unique.info()

<class 'pandas.core.frame.DataFrame'>
Index: 215700 entries, 0 to 256983
Data columns (total 19 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   Loan ID                       215700 non-null  object 
 1   Customer ID                   215700 non-null  object 
 2   Loan Status                   215700 non-null  object 
 3   Current Loan Amount           215700 non-null  int64  
 4   Term                          215700 non-null  object 
 5   Credit Score                  160692 non-null  float64
 6   Years in current job          206710 non-null  object 
 7   Home Ownership                215700 non-null  object 
 8   Annual Income                 160692 non-null  float64
 9   Purpose                       215700 non-null  object 
 10  Monthly Debt                  215700 non-null  object 
 11  Years of Credit History       215700 non-null  float64
 12  Months since last delinquent  97438 non-null   fl

In [28]:
df_unique['Years in current job'] = pd.to_numeric(df_unique['Years in current job'], errors='coerce').astype('Int64')

In [29]:
df_unique['Monthly Debt'].str.contains('$', case=False, na=False).sum()

215700

In [30]:
df_unique['Monthly Debt'] = df_unique['Monthly Debt'].str[1:]# delete $ sign

In [31]:
df_unique['Monthly Debt']

0            584.03
1         1,106.04 
2         1,321.85 
3            751.92
4            355.18
            ...    
256977       982.82
256979    1,706.58 
256980    1,376.47 
256981       297.96
256983    2,525.82 
Name: Monthly Debt, Length: 215700, dtype: object

In [32]:
df_unique['Monthly Debt'] = df_unique['Monthly Debt'].str.replace(',', '', regex=False)

In [33]:

df_unique['Monthly Debt'] = pd.to_numeric(df_unique['Monthly Debt'], errors='coerce').astype('float')
df_unique['Maximum Open Credit'] = pd.to_numeric(df_unique['Maximum Open Credit'], errors='coerce').astype('Int64')

In [34]:
df_unique['Monthly Debt'] = pd.to_numeric(df_unique['Monthly Debt'], errors='coerce').astype('float')

In [35]:
for col in df_unique.select_dtypes(include=['object']).columns:
    df_unique[col] = df_unique[col].astype('category') 

In [36]:
print(df_unique.dtypes)

Loan ID                         category
Customer ID                     category
Loan Status                     category
Current Loan Amount                int64
Term                            category
Credit Score                     float64
Years in current job               Int64
Home Ownership                  category
Annual Income                    float64
Purpose                         category
Monthly Debt                     float64
Years of Credit History          float64
Months since last delinquent     float64
Number of Open Accounts            int64
Number of Credit Problems          int64
Current Credit Balance             int64
Maximum Open Credit                Int64
Bankruptcies                     float64
Tax Liens                        float64
dtype: object


In [37]:
(df_unique['Credit Score'] > 800).sum()

14433

In [38]:
c=df_unique[df_unique['Credit Score'] > 800]

In [39]:
c["Loan Status"].value_counts()

Loan Status
Charged Off    14433
Fully Paid         0
Name: count, dtype: int64

In [40]:
df_unique = df_unique[df_unique['Credit Score'] <= 800]

In [41]:
df_unique = df_unique.reset_index(drop=True)

In [42]:
imp=mf.ImputationKernel(df_unique)

In [43]:
df_unique = imp.complete_data(0)

In [44]:
df_unique.isnull().sum()

Loan ID                         0
Customer ID                     0
Loan Status                     0
Current Loan Amount             0
Term                            0
Credit Score                    0
Years in current job            0
Home Ownership                  0
Annual Income                   0
Purpose                         0
Monthly Debt                    0
Years of Credit History         0
Months since last delinquent    0
Number of Open Accounts         0
Number of Credit Problems       0
Current Credit Balance          0
Maximum Open Credit             0
Bankruptcies                    0
Tax Liens                       0
dtype: int64

In [45]:
#Finally we

In [46]:
df_unique.sample(5)

,Loan ID,Customer ID,Loan Status,Current Loan Amount,Term,Credit Score,Years in current job,Home Ownership,Annual Income,Purpose,Monthly Debt,Years of Credit History,Months since last delinquent,Number of Open Accounts,Number of Credit Problems,Current Credit Balance,Maximum Open Credit,Bankruptcies,Tax Liens
108692,be1dfbe3-d3b7-44c1-93ec-0c68a6f2d0a8,12c3f9bd-c63b-4def-a996-ce5784cda4d2,Fully Paid,33990,Short Term,743.0,15,Home Mortgage,124963.0,Debt Consolidation,1749.48,26.5,21.0,15,0,35048,48277,0.0,0.0
85370,957308ee-2f99-4fb3-909c-7d49b833566a,378819ad-1967-482e-80ce-1bdcf9f4ec63,Fully Paid,5901,Short Term,740.0,15,Home Mortgage,122938.0,Debt Consolidation,301.20,14.6,37.0,13,0,12129,19657,0.0,0.0
127962,df9ae7ca-d33b-4abc-b2ec-c6455f7a5cc6,377a2055-0d25-4dd0-a7c9-28cb61f68259,Charged Off,8861,Short Term,742.0,15,Rent,63802.0,Debt Consolidation,1329.21,27.4,19.0,13,1,9284,15447,1.0,0.0
117015,ccc20c38-36aa-4d40-a7bf-118db80ffb5a,f2b85114-a1e6-42ad-9f36-8100f16dc2a0,Fully Paid,20498,Long Term,720.0,0,Home Mortgage,73793.0,Home Improvements,395.41,19.2,17.0,12,0,18140,28794,0.0,0.0
76112,858d9d82-06da-4432-8eb7-346cb64c8466,b24a79f3-f524-4c7b-98f5-846f1fc94522,Fully Paid,34526,Short Term,719.0,15,Rent,89100.0,Debt Consolidation,703.15,17.4,31.0,7,0,15387,20598,0.0,0.0


In [47]:
#get dummies

In [48]:
df_unique=df_unique.drop(['Loan ID','Customer ID'],axis=1)

In [49]:
df_unique.corr(numeric_only=True)

,Current Loan Amount,Credit Score,Years in current job,Annual Income,Monthly Debt,Years of Credit History,Months since last delinquent,Number of Open Accounts,Number of Credit Problems,Current Credit Balance,Maximum Open Credit,Bankruptcies,Tax Liens
Current Loan Amount,1.000000,0.026329,0.004675,0.008478,0.002896,0.011048,-0.000864,0.002489,0.001049,0.003100,0.004836,0.003050,-0.001487
Credit Score,0.026329,1.000000,-0.008683,0.012500,-0.082631,0.082433,0.018657,-0.033762,-0.070287,-0.017130,0.009316,-0.058059,-0.028152
Years in current job,0.004675,-0.008683,1.000000,0.029600,0.056268,0.091913,0.001626,0.020586,0.021773,0.038422,-0.004734,0.022733,0.003301
Annual Income,0.008478,0.012500,0.029600,1.000000,0.460396,0.153411,-0.024271,0.141479,-0.020128,0.304878,0.030958,-0.048412,0.032560
Monthly Debt,0.002896,-0.082631,0.056268,0.460396,1.000000,0.194198,-0.021379,0.411377,-0.058245,0.479588,0.020544,-0.080648,0.013802
Years of Credit History,0.011048,0.082433,0.091913,0.153411,0.194198,1.000000,-0.017094,0.132538,0.058825,0.208983,0.018946,0.060041,0.019782
Months since last delinquent,-0.000864,0.018657,0.001626,-0.024271,-0.021379,-0.017094,1.000000,-0.017098,0.044916,-0.007583,0.001029,0.054616,-0.001727
Number of Open Accounts,0.002489,-0.033762,0.020586,0.141479,0.411377,0.132538,-0.017098,1.000000,-0.017919,0.226939,0.014392,-0.025583,0.002012
Number of Credit Problems,0.001049,-0.070287,0.021773,-0.020128,-0.058245,0.058825,0.044916,-0.017919,1.000000,-0.111467,-0.008011,0.766184,0.579506
Current Credit Balance,0.003100,-0.017130,0.038422,0.304878,0.479588,0.208983,-0.007583,0.226939,-0.111467,1.000000,0.098260,-0.125971,-0.013520


In [50]:
print(df_unique.dtypes)

Loan Status                     category
Current Loan Amount                int64
Term                            category
Credit Score                     float64
Years in current job               Int64
Home Ownership                  category
Annual Income                    float64
Purpose                         category
Monthly Debt                     float64
Years of Credit History          float64
Months since last delinquent     float64
Number of Open Accounts            int64
Number of Credit Problems          int64
Current Credit Balance             int64
Maximum Open Credit                Int64
Bankruptcies                     float64
Tax Liens                        float64
dtype: object


In [51]:
df_unique["Loan Status"].value_counts()

Loan Status
Fully Paid     133982
Charged Off     12277
Name: count, dtype: int64

In [52]:
d={"Charged Off":0,"Fully Paid":1}#classification

In [53]:
df_unique["Loan Status"].value_counts()

Loan Status
Fully Paid     133982
Charged Off     12277
Name: count, dtype: int64

In [54]:
df_unique.sample()

,Loan Status,Current Loan Amount,Term,Credit Score,Years in current job,Home Ownership,Annual Income,Purpose,Monthly Debt,Years of Credit History,Months since last delinquent,Number of Open Accounts,Number of Credit Problems,Current Credit Balance,Maximum Open Credit,Bankruptcies,Tax Liens
28688,Fully Paid,16008,Long Term,729.0,15,Home Mortgage,88585.0,Debt Consolidation,1077.79,14.1,79.0,19,0,12693,15808,0.0,0.0


In [55]:
df_unique.isnull().sum()

Loan Status                     0
Current Loan Amount             0
Term                            0
Credit Score                    0
Years in current job            0
Home Ownership                  0
Annual Income                   0
Purpose                         0
Monthly Debt                    0
Years of Credit History         0
Months since last delinquent    0
Number of Open Accounts         0
Number of Credit Problems       0
Current Credit Balance          0
Maximum Open Credit             0
Bankruptcies                    0
Tax Liens                       0
dtype: int64

In [56]:
x=df_unique.drop("Loan Status", axis=1)
y=df_unique[["Loan Status"]]

In [57]:
x=pd.get_dummies(x,drop_first=True)

In [58]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.20,random_state=42)

In [59]:
b=BernoulliNB()

In [60]:
b.fit(x_train,y_train)

BernoulliNB()

In [61]:
bpred=b.predict(x_test)

In [62]:
y.value_counts()

Loan Status
Fully Paid     133982
Charged Off     12277
Name: count, dtype: int64

In [63]:
accuracy_score(bpred,y_test)

0.9139204156980719

In [64]:
classification_test(x, y)

(                     Accuracy
 Random Forest        0.913955
 BernoulliNB           0.91392
 Logistic Regression  0.913886
 Gradient Boosting    0.913852
 K-Neighbors          0.906297
 Decision Tree        0.849378
 GaussianNB           0.330644,
 {'BernoulliNB': {'Charged Off': {'precision': 0.0,
    'recall': 0.0,
    'f1-score': 0.0,
    'support': 2518.0},
   'Fully Paid': {'precision': 0.9139204156980719,
    'recall': 1.0,
    'f1-score': 0.9550244704033151,
    'support': 26734.0},
   'accuracy': 0.9139204156980719,
   'macro avg': {'precision': 0.45696020784903596,
    'recall': 0.5,
    'f1-score': 0.47751223520165753,
    'support': 29252.0},
   'weighted avg': {'precision': 0.8352505262297365,
    'recall': 0.9139204156980719,
    'f1-score': 0.8728163609928287,
    'support': 29252.0}},
  'GaussianNB': {'Charged Off': {'precision': 0.11324689455072989,
    'recall': 0.9920571882446386,
    'f1-score': 0.20328776041666666,
    'support': 2518.0},
   'Fully Paid': {'precisi

In [130]:
print(classification_report(bpred,y_test))

              precision    recall  f1-score   support

 Charged Off       0.00      0.00      0.00         0
  Fully Paid       1.00      0.91      0.96     29252

    accuracy                           0.91     29252
   macro avg       0.50      0.46      0.48     29252
weighted avg       1.00      0.91      0.96     29252



In [132]:
#f1 score and accuracy score both is above 90 percent so it is good model